# 01. 데이터 확인
라벨(Validation)은 압축을 풀지 않고 `VL.zip`을 zipfile로 읽는다 (작은 JSON이 많아 /mnt/c에서 풀면 느림).
사진(`VS.zip`)은 아직 없음. 전체 집계는 `src/label_stats.py` → `docs/label_stats.md`.

In [ ]:
from pathlib import Path
import json
import zipfile
from collections import Counter

DATA_DIR  = Path("../data")
IMG_DIR   = DATA_DIR / "images"                      # VS.zip 압축 해제 위치 (사진 아직 없음)
LABEL_ZIP = DATA_DIR / "02.라벨링데이터" / "VL.zip"
IMG_EXTS  = {".jpg", ".jpeg", ".png"}
# 클래스는 JSON의 annotations[i]["category_id"] -> categories[id]["name"]. 폴더명(zip 내부 2단계)과도 같음.

## zip 내부 폴더 구조 (깊이 2)

In [ ]:
z = zipfile.ZipFile(LABEL_ZIP)
label_names = [n for n in z.namelist() if n.endswith(".json")]
print(len(label_names), "label files")
Counter("/".join(n.split("/")[:2]) for n in label_names)

## 사진 수 (사진을 받은 뒤)

In [ ]:
imgs = [p for p in IMG_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS] if IMG_DIR.exists() else []
print(len(imgs), "images")

## 라벨 샘플 1개 (environment 시계열은 생략)

In [ ]:
sample = json.loads(z.read(label_names[0]))
print(label_names[0])
sample["environment"] = {k: f"<{len(v)} values>" if isinstance(v, list) else v for k, v in sample["environment"].items()}
print(json.dumps(sample, ensure_ascii=False, indent=2))

## 클래스별 개수 (폴더명 기준, 이미지 1장 = JSON 1개)

In [ ]:
counts = Counter(n.split("/")[1] for n in label_names)
counts